## 1. Import Dataset dan Library
Mengimport library yang dibutuhkan kemudian mengimport dataset yang sudah didownload menggunakan pd.read_excel. Setelah itu melakukan sedikit eksplorasi data dengan menampilkan head, info, shape.

In [ ]:
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
from IPython.display import display
pd.set_option('display.max_colwidth', None)

In [2]:
df = pd.read_excel('Online Retail.xlsx')
display(df.head())

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
print('Shape:', df.shape)
print()
df.info()

Shape: (541909, 8)

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


## 2. Eksplorasi dan Pembersihan Data

**Kolom yang relevan.** Pola berurutan disusun per pelanggan, sehingga kolom yang dipakai adalah:
- `CustomerID` → pemilik sequence (satu pelanggan = satu sequence)
- `InvoiceDate` → mengurutkan pembelian secara waktu
- `InvoiceNo` → pembentuk itemset (semua produk dalam satu invoice = satu itemset)
- `Description` → nama produk (item)
- `Quantity` → dipakai untuk menyaring baris tidak valid

Pembersihan dilakukan sesuai ketentuan: hapus `Description` kosong, invoice dibatalkan (diawali "C"), `Quantity` negatif atau nol, dan `CustomerID` kosong. Jumlah baris, invoice, dan pelanggan dicatat sebelum dan sesudah setiap tahap.

In [5]:
print('Nilai kosong per kolom:')
print(df.isna().sum())
print()
print('Jumlah invoice unik  :', df['InvoiceNo'].nunique())
print('Jumlah pelanggan unik:', df['CustomerID'].nunique())
print('Jumlah produk unik   :', df['Description'].nunique())
print('Invoice dibatalkan   :', df['InvoiceNo'].astype(str).str.startswith('C').sum(), 'baris')
print('Quantity <= 0        :', (df['Quantity'] <= 0).sum(), 'baris')
print('Rentang tanggal      :', df['InvoiceDate'].min(), 's.d.', df['InvoiceDate'].max())
print()
display(df[['Quantity', 'UnitPrice']].describe().round(2))

Nilai kosong per kolom:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Jumlah invoice unik  : 25900
Jumlah pelanggan unik: 4372
Jumlah produk unik   : 4223
Invoice dibatalkan   : 9288 baris
Quantity <= 0        : 10624 baris
Rentang tanggal      : 2010-12-01 08:26:00 s.d. 2011-12-09 12:50:00



,Quantity,UnitPrice
count,541909.00,541909.00
mean,9.55,4.61
std,218.08,96.76
min,-80995.00,-11062.06
25%,1.00,1.25
50%,3.00,2.08
75%,10.00,4.13
max,80995.00,38970.00


In [4]:
log = []
def catat(tahap, d):
    log.append({'Tahap': tahap, 'Jumlah baris': len(d), 'Jumlah invoice': d['InvoiceNo'].nunique(),
                'Jumlah pelanggan': d['CustomerID'].nunique()})

data = df.copy()
data['InvoiceNo'] = data['InvoiceNo'].astype(str)
catat('Data awal', data)

# Hapus baris yang memiliki Description kosong
data['Description'] = data['Description'].astype('object').str.strip()
data = data[data['Description'].notna() & (data['Description'] != '')]
catat('Hapus Description kosong', data)

# Hapus invoice dibatalkan (C)
data = data[~data['InvoiceNo'].str.startswith('C')]
catat('Hapus invoice dibatalkan (C)', data)

# Hapus baris dengan Quantity <= 0
data = data[data['Quantity'] > 0]
catat('Hapus Quantity <= 0', data)

# Hapus baris tanpa CustomerID
data = data[data['CustomerID'].notna()].copy()
data['CustomerID'] = data['CustomerID'].astype(int)
catat('Hapus CustomerID kosong', data)

ringkasan = pd.DataFrame(log)
ringkasan['Baris terhapus'] = (-ringkasan['Jumlah baris'].diff()).fillna(0).astype(int)
ringkasan['Invoice terhapus'] = (-ringkasan['Jumlah invoice'].diff()).fillna(0).astype(int)
display(ringkasan)

print(f"Sebelum : {len(df):,} baris | {df['InvoiceNo'].nunique():,} invoice | {df['CustomerID'].nunique():,} pelanggan")
print(f"Sesudah : {len(data):,} baris | {data['InvoiceNo'].nunique():,} invoice | {data['CustomerID'].nunique():,} pelanggan")

,Tahap,Jumlah baris,Jumlah invoice,Jumlah pelanggan,Baris terhapus,Invoice terhapus
0,Data awal,541909,25900,4372,0,0
1,Hapus Description kosong,540454,24445,4372,1455,1455
2,Hapus invoice dibatalkan (C),531166,20609,4339,9288,3836
3,Hapus Quantity <= 0,530693,20136,4339,473,473
4,Hapus CustomerID kosong,397924,18536,4339,132769,1600


Sebelum : 541,909 baris | 25,900 invoice | 4,372 pelanggan
Sesudah : 397,924 baris | 18,536 invoice | 4,339 pelanggan


## 3. Persiapan Data untuk Sequential Pattern Mining

Data diurutkan berdasarkan `CustomerID` dan `InvoiceDate`, lalu item pada satu invoice dikelompokkan menjadi satu itemset. Setiap pelanggan menjadi satu sequence berisi daftar itemset yang urut secara waktu, disimpan sebagai list of list.

In [7]:
data = data.sort_values(['CustomerID', 'InvoiceDate', 'InvoiceNo'])

invoice = (data.groupby(['CustomerID', 'InvoiceNo'], sort=False)
               .agg(waktu=('InvoiceDate', 'min'), items=('Description', lambda s: sorted(set(s))))
               .reset_index()
               .sort_values(['CustomerID', 'waktu', 'InvoiceNo']))

seq_pelanggan = invoice.groupby('CustomerID', sort=True)['items'].agg(list)
sequences = seq_pelanggan.tolist()      # list of list of list
N = len(sequences)

print('Jumlah sequence (pelanggan):', N)
print(f'Pelanggan {seq_pelanggan.index[1]} memiliki {len(sequences[1])} itemset, itemset pertama:')
print(sequences[1][0][:5])

Jumlah sequence (pelanggan): 4339
Pelanggan 12347 memiliki 7 itemset, itemset pertama:
['3D DOG PICTURE PLAYING CARDS', 'AIRLINE BAG VINTAGE JET SET BROWN', 'ALARM CLOCK BAKELIKE CHOCOLATE', 'ALARM CLOCK BAKELIKE GREEN', 'ALARM CLOCK BAKELIKE ORANGE']


## 4. Penerapan Algoritma PrefixSpan

Pustaka `prefixspan` memperlakukan sequence sebagai deretan item tunggal sehingga batas invoice hilang (dua produk dalam satu invoice akan terbaca berurutan). Karena itu PrefixSpan diimplementasikan sendiri dengan logika yang sama (*prefix-projection*), tetapi tetap menjaga batas itemset. Pola `A → B` berarti A dibeli pada suatu invoice dan B dibeli pada invoice yang lebih akhir.

Minimum support dihitung relatif terhadap jumlah pelanggan, dengan titik awal 0,01.

In [ ]:
def prefixspan(seqs, min_count, max_len=6):
    hasil = {}
    def tumbuh(pola, proyeksi):
        if len(pola) >= max_len:
            return
        kandidat = defaultdict(dict) 
        for sid, mulai in proyeksi:
            s, terlihat = seqs[sid], {}
            for i in range(mulai, len(s)):
                for it in s[i]:
                    terlihat.setdefault(it, i)
            for it, i in terlihat.items():
                kandidat[it][sid] = i + 1
        for it, m in kandidat.items():
            if len(m) >= min_count:
                p = pola + (it,)
                hasil[p] = len(m)
                tumbuh(p, list(m.items()))
    tumbuh((), [(i, 0) for i in range(len(seqs))])
    return hasil

**Penyesuaian minimum support.** Jumlah pola pada beberapa nilai minimum support dibandingkan (panjang pola dibatasi 3 agar perbandingan cepat dihitung).

In [9]:
rows = []
for ms in [0.01, 0.02, 0.03, 0.05]:
    r = prefixspan(sequences, ms * N, max_len=3)
    c = Counter(len(p) for p in r)
    rows.append({'min_support': ms, 'Minimal pelanggan': int(np.ceil(ms * N)), 'Pola panjang 1': c[1],
                 'Pola panjang 2': c[2], 'Pola panjang 3': c[3], 'Total': len(r)})
display(pd.DataFrame(rows))

,min_support,Minimal pelanggan,Pola panjang 1,Pola panjang 2,Pola panjang 3,Total
0,0.01,44,1763,15707,17262,34732
1,0.02,87,1046,1506,417,2969
2,0.03,131,657,281,9,947
3,0.05,217,265,8,1,274


Pada `min_support` 0,01 (minimal 44 pelanggan) ditemukan 15.707 pola panjang 2 dan 17.262 pola panjang 3 (total 34.732 pola, itu pun baru sampai panjang 3). Jumlah ini terlalu banyak untuk dievaluasi dan dibaca satu per satu. Sebaliknya, pada 0,05 hanya ada 9 pola berurutan, terlalu sedikit. Nilai yang dipilih adalah **`min_support = 0,03`** (minimal 131 pelanggan), yang menghasilkan 291 pola berurutan.

In [10]:
MIN_SUP = 0.03
freq = prefixspan(sequences, MIN_SUP * N)
print(f'min_support = {MIN_SUP} (minimal {int(np.ceil(MIN_SUP * N))} pelanggan)')
print('Jumlah pola ditemukan:', len(freq))
display(pd.Series(Counter(len(p) for p in freq)).sort_index().rename('Jumlah pola').to_frame().rename_axis('Panjang pola'))

min_support = 0.03 (minimal 131 pelanggan)
Jumlah pola ditemukan: 948


,Jumlah pola
Panjang pola,
1,657
2,281
3,9
4,1


## 5. Evaluasi Pola Berurutan

In [11]:
rows = []
for p, c in freq.items():
    if len(p) >= 2:
        rows.append((' → '.join(p), len(p), c, c / N, c / freq[p[:-1]]))
pola = (pd.DataFrame(rows, columns=['Pola berurutan', 'Panjang', 'Jumlah pelanggan', 'support', 'confidence'])
          .sort_values(['support', 'confidence'], ascending=False, kind='mergesort')
          .reset_index(drop=True))

print('Jumlah pola berurutan:', len(pola))
print('20 pola berurutan dengan support tertinggi:')
display(pola.head(20).round(3))

Jumlah pola berurutan: 291
20 pola berurutan dengan support tertinggi:


,Pola berurutan,Panjang,Jumlah pelanggan,support,confidence
0,WHITE HANGING HEART T-LIGHT HOLDER → WHITE HANGING HEART T-LIGHT HOLDER,2,405,0.093,0.473
1,REGENCY CAKESTAND 3 TIER → REGENCY CAKESTAND 3 TIER,2,330,0.076,0.375
2,JUMBO BAG RED RETROSPOT → JUMBO BAG RED RETROSPOT,2,312,0.072,0.491
3,ASSORTED COLOUR BIRD ORNAMENT → ASSORTED COLOUR BIRD ORNAMENT,2,291,0.067,0.429
4,PARTY BUNTING → PARTY BUNTING,2,280,0.065,0.395
5,LUNCH BAG RED RETROSPOT → LUNCH BAG RED RETROSPOT,2,276,0.064,0.519
6,LUNCH BAG BLACK SKULL. → LUNCH BAG BLACK SKULL.,2,229,0.053,0.500
7,SET OF 3 CAKE TINS PANTRY DESIGN → SET OF 3 CAKE TINS PANTRY DESIGN,2,225,0.052,0.352
8,WHITE HANGING HEART T-LIGHT HOLDER → WHITE HANGING HEART T-LIGHT HOLDER → WHITE HANGING HEART T-LIGHT HOLDER,3,222,0.051,0.548
9,LUNCH BAG RED RETROSPOT → LUNCH BAG BLACK SKULL.,2,212,0.049,0.398


In [12]:
top3 = pola.head(3)
print('Tiga pola terbaik berdasarkan support:')
display(top3.round(3))

Tiga pola terbaik berdasarkan support:


,Pola berurutan,Panjang,Jumlah pelanggan,support,confidence
0,WHITE HANGING HEART T-LIGHT HOLDER → WHITE HANGING HEART T-LIGHT HOLDER,2,405,0.093,0.473
1,REGENCY CAKESTAND 3 TIER → REGENCY CAKESTAND 3 TIER,2,330,0.076,0.375
2,JUMBO BAG RED RETROSPOT → JUMBO BAG RED RETROSPOT,2,312,0.072,0.491
